## **Destination Recommendation System**

In [62]:
import pandas as pd
import numpy as np

In [63]:
data = pd.read_csv('/kaggle/input/datasets/dheerajsingh28b/uttrakhand-destination/destination.csv')

In [64]:
data.head(3)

,destination_id,name,city,district,state,category,description,interest_tags,rating,review_count,sentiment_score,entry_fee_inr,estimated_daily_budget_inr,ideal_duration_days,latitude,longitude,best_season,crowd_level
0,DEST_00001,Robber's Cave (Guchhupani),Dehradun,Dehradun,Uttarakhand,Nature,A narrow limestone gorge near Anarwala village...,"guchhupani, limestone gorge, stream wading, an...",4.1,18420,0.742,35,1480,1,30.3761,78.0604,Summer,High
1,DEST_00002,Forest Research Institute (FRI),Dehradun,Dehradun,Uttarakhand,Heritage,Greco-Roman colonial architecture spread acros...,"colonial architecture, forestry museum, chakra...",4.6,24310,0.894,40,1620,1,30.3437,77.9996,Winter,Moderate
2,DEST_00003,Mindrolling Monastery,Clement Town,Dehradun,Uttarakhand,Spiritual,One of India's largest Buddhist centers featur...,"tibetan buddhism, great stupa, clement town, m...",4.5,14980,0.868,0,1540,1,30.2618,78.0055,Autumn,Moderate


In [65]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830 entries, 0 to 829
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   destination_id              830 non-null    object 
 1   name                        830 non-null    object 
 2   city                        830 non-null    object 
 3   district                    830 non-null    object 
 4   state                       830 non-null    object 
 5   category                    830 non-null    object 
 6   description                 830 non-null    object 
 7   interest_tags               830 non-null    object 
 8   rating                      830 non-null    float64
 9   review_count                830 non-null    int64  
 10  sentiment_score             830 non-null    float64
 11  entry_fee_inr               830 non-null    int64  
 12  estimated_daily_budget_inr  830 non-null    int64  
 13  ideal_duration_days         830 non

In [66]:
data['cleaned_text'] = data['city'] + " " + data['description'] + " " + data['interest_tags']

In [67]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Download necessary NLTK data (Only runs once)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [68]:
# Initialize tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [69]:
# 2. Define the Text Preprocessing Function
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove numbers and special characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize (split into words)
    tokens = nltk.word_tokenize(text)
    # Remove stopwords and Lemmatize
    clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(clean_tokens)

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Build the TF-IDF Matrix (This is sparse and uses very little memory)
print("🔢 Building TF-IDF Matrix...")
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(data['cleaned_text'])

print(f"✅ Matrix shape: {tfidf_matrix.shape} (Rows: Destinations, Columns: Unique Words)")

🔢 Building TF-IDF Matrix...
✅ Matrix shape: (830, 3210) (Rows: Destinations, Columns: Unique Words)


In [74]:
# 2. Memory-Safe Recommendation Function
def get_similar_destinations_safe(dest_name, top_n=5, threshold=0.1):
    """
    Finds similar destinations WITHOUT creating a massive 20GB matrix.
    """
    # Check if destination exists
    if dest_name not in data['name'].values:
        return f"❌ Destination '{dest_name}' not found."
    
    # Get the index of the requested destination
    idx = data[data['name'] == dest_name].index[0]
    
    # Extract ONLY the vector for this specific destination (1 row)
    dest_vector = tfidf_matrix[idx]
    
    # Calculate similarity between THIS destination and ALL others
    # This creates a tiny 1 x 50,000 array instead of a 50,000 x 50,000 grid!
    sim_scores = cosine_similarity(dest_vector, tfidf_matrix).flatten()
    
    # Get indices of destinations that pass the threshold (excluding itself)
    valid_indices = np.where((sim_scores > threshold) & (np.arange(len(sim_scores)) != idx))[0]
    
    # Sort these valid indices by their similarity score (highest first)
    sorted_indices = valid_indices[np.argsort(sim_scores[valid_indices])[::-1]]
    
    # Pick the top N
    top_indices = sorted_indices[:top_n]
    
    if len(top_indices) == 0:
        return f"⚠️ No destinations found above {threshold} threshold."
    
    # Create the final clean DataFrame
    result_df = data[['name', 'city', 'category', 'rating', 'interest_tags', 'estimated_daily_budget_inr']].iloc[top_indices].copy()
    result_df['similarity_score'] = [round(sim_scores[i], 4) for i in top_indices]
    
    return result_df.reset_index(drop=True)

In [83]:
# 🧪 TEST 1: Nature/Adventure
get_similar_destinations_safe("Robber's Cave (Guchhupani)", top_n=8, threshold=0.05)

,name,city,category,rating,interest_tags,estimated_daily_budget_inr,similarity_score
0,Dehradun Sahastradhara to Sera Village Waterfa...,Sera Village,Adventure,4.4,"sera village hike, upper sahastradhara, limest...",1640,0.2167
1,Dehradun Robber's Cave to Tapkeshwar Cantonmen...,Dehradun,Nature,4.2,"guchhupani to tapkeshwar, garhi cantt drive, s...",1480,0.1752
2,Padampuri Waterfall & River Pools,Padampuri,Nature,4.4,"padampuri falls, crystal river pools, dhari ro...",2350,0.1486
3,Patal Bhuvaneshwar Limestone Cave,Gangolihat,Spiritual,4.8,"patal bhuvaneshwar, subterranean cave, gangoli...",2190,0.1448
4,Bhatta Falls,Bhatta Village,Nature,4.1,"bhatta village, natural pools, bamboo bridge, ...",2190,0.1397
5,Gauri Udiyar Cave Temple,Bageshwar,Spiritual,4.4,"gauri udiyar, limestone cave shrine, stalactit...",1790,0.1158
6,Jageshwar Dham Temple Complex,Jageshwar,Spiritual,4.8,"jageshwar temples, 125 stone shrines, jatagang...",2240,0.1156
7,Budher Caves (Miola Caves),Chakrata,Adventure,4.5,"limestone caverns, budher meadow, alpine pastu...",2560,0.1106


In [82]:
# 🧪 TEST 2: Spiritual
get_similar_destinations_safe("Mindrolling Monastery", top_n=7, threshold=0.05)

,name,city,category,rating,interest_tags,estimated_daily_budget_inr,similarity_score
0,Dehradun Clement Town Buddha Temple to Rajaji ...,Clement Town,Nature,4.2,"clement town forest walk, shivalik fringe, sal...",1490,0.3741
1,Chaukori Tea Estates & Musk Deer Park,Chaukori,Nature,4.6,"chaukori tea gardens, musk deer farm, panchach...",2380,0.1217
2,Naina Devi Temple,Nainital,Spiritual,4.7,"naina devi shakti peeth, northern lake shore, ...",2890,0.1173
3,Lansdowne Cantonment & Bhulla Tal,Lansdowne,Nature,4.4,"bhulla tal, garhwal rifles, cantonment town, p...",2540,0.1055
4,Beatles Ashram (Chaurasi Kutia),Rishikesh,Heritage,4.4,"chaurasi kutia, graffiti murals, 1968 beatles ...",2250,0.1005
5,Pithoragarh Fort (Baurari Fort),Pithoragarh,Heritage,4.4,"pithoragarh fort, gorkha architecture, soar va...",1920,0.0948
6,Ranikhet Chaubatia Orchards,Ranikhet,Nature,4.4,"chaubatia gardens, apple orchards, nanda devi ...",2450,0.0789


In [81]:
# 🧪 TEST 3: Heritage
get_similar_destinations_safe("Forest Research Institute (FRI)", top_n=10, threshold=0.1)

,name,city,category,rating,interest_tags,estimated_daily_budget_inr,similarity_score
0,Dehradun Dehradun to Chakrata-Mussoorie-Landou...,Chakrata,Nature,4.6,"doon hills loop, chakrata mussoorie circuit, d...",2480,0.2287
1,Ram Tal Horticultural Garden,Chakrata,Family,4.2,"ram tal garden, chakrata picnic, apple orchard...",2180,0.1646
2,Mukteshwar Indian Veterinary Research Institut...,Mukteshwar,Heritage,4.5,"ivri mukteshwar, 1893 colonial campus, deodar ...",2840,0.1610
3,Thangdhar & Kanasar Pine Sanctuary,Chakrata,Nature,4.6,"thangdhar camp, chakrata offbeat, silent deoda...",2380,0.1295
4,Kimona Falls & Chilmiri Neck Sunset Point,Chakrata,Nature,4.5,"chilmiri neck, kimona falls, chakrata sunset, ...",2240,0.1201
5,Dehradun Subhash Road & Astley Hall Heritage P...,Dehradun,Heritage,4.2,"astley hall, colonial arcades, vintage booksho...",1580,0.1193
6,Mussoorie Mussoorie to Chakrata Yamuna-Kempty ...,Lakhwar,Nature,4.6,"mussoorie to chakrata road, lakhwar valley, ya...",2380,0.1005
7,Mussoorie Landour Sister's Bazaar to Dhanaulti...,Buranskhanda,Nature,4.6,"landour to buranskhanda, colonial to orchards,...",2740,0.1004
